# Database Expansion

This notebook orchestrates synthetic customer data generation. The generation logic lives in `../src/big_query_functions.py`, while this notebook loads the source datasets, creates a large synthetic DataFrame, writes it to Parquet, and checks the output shape and quality.

## Import Required Libraries

This cell imports the basic Python tools used in the notebook and loads the reusable synthetic data functions from `src/big_query_functions.py`. Keeping the logic in a separate module makes the notebook cleaner and easier to maintain.

In [25]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append('../src')

from big_query_functions import (
    generate_synthetic_campaign_data,
    get_synthetic_output_columns,
    prepare_generation_metadata,
    summarize_group_proportions,
    validate_synthetic_campaign_data,
    write_synthetic_dataset,
)
from custom_functions import error_handling

## Define Input and Output Paths

This cell centralizes all file paths and configuration values. The notebook reads the cleaned customer data, grouped statistics, and original raw schema, then writes the expanded dataset as a local Parquet file.

In [27]:
DATA_DIR = Path('../data')

TREATED_DATA_PATH = DATA_DIR / 'treated_data.csv'
GROUPED_MEAN_PATH = DATA_DIR / 'grouped_data_mean.csv'
GROUPED_STD_PATH = DATA_DIR / 'grouped_data_std.csv'
RAW_CAMPAIGN_PATH = DATA_DIR / 'marketing_campaign.csv'
SYNTHETIC_OUTPUT_PATH = DATA_DIR / 'synthetic_marketing_campaign.parquet'

N_SYNTHETIC_ROWS = 1_000_000
RANDOM_STATE = 42

## Load Source Datasets

This cell loads the data needed to create the synthetic customer base: cleaned customer records, group-level means, group-level standard deviations, and the original campaign file used as the reference schema.

In [29]:
try:
    df_treated_data = pd.read_csv(TREATED_DATA_PATH)
    df_grouped_mean_data = pd.read_csv(GROUPED_MEAN_PATH)
    df_grouped_std_data = pd.read_csv(GROUPED_STD_PATH)
    df_raw_campaign = pd.read_csv(RAW_CAMPAIGN_PATH, sep='\t')

    print('Data loaded')
    print(f'Treated data: {df_treated_data.shape}')
    print(f'Grouped means: {df_grouped_mean_data.shape}')
    print(f'Grouped stds: {df_grouped_std_data.shape}')
    print(f'Raw campaign: {df_raw_campaign.shape}')
except Exception as e:
    error_handling(e)

Data loaded
Treated data: (2237, 31)
Grouped means: (25, 27)
Grouped stds: (25, 27)
Raw campaign: (2240, 29)


## Prepare Generation Metadata

This cell prepares the rules used by the generator: expected output columns, customer group proportions, date limits, numeric bounds, and fallback behavior for small groups. The output columns intentionally exclude `Z_CostContact` and `Z_Revenue`.

In [31]:
generation_metadata = prepare_generation_metadata(
    df_treated_data,
    df_grouped_mean_data,
    df_grouped_std_data,
    df_raw_campaign,
)

get_synthetic_output_columns(df_raw_campaign)

['ID',
 'Year_Birth',
 'Education',
 'Marital_Status',
 'Income',
 'Kidhome',
 'Teenhome',
 'Dt_Customer',
 'Recency',
 'MntWines',
 'MntFruits',
 'MntMeatProducts',
 'MntFishProducts',
 'MntSweetProducts',
 'MntGoldProds',
 'NumDealsPurchases',
 'NumWebPurchases',
 'NumCatalogPurchases',
 'NumStorePurchases',
 'NumWebVisitsMonth',
 'AcceptedCmp3',
 'AcceptedCmp4',
 'AcceptedCmp5',
 'AcceptedCmp1',
 'AcceptedCmp2',
 'Complain',
 'Response']

## Generate Synthetic Customer Records

This cell creates 1,000,000 synthetic customers. It preserves the original distribution of `Marital_Status` and `Education`, samples numeric fields using grouped means and standard deviations, and adds controlled noise to make the records realistic but not duplicated.

In [33]:
df_synthetic_campaign = generate_synthetic_campaign_data(
    df_treated_data,
    df_grouped_mean_data,
    df_grouped_std_data,
    df_raw_campaign,
    n_rows=N_SYNTHETIC_ROWS,
    random_state=RANDOM_STATE,
    metadata=generation_metadata,
)

df_synthetic_campaign.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response
0,1,1982,Graduation,Together,41952.61,1,0,2012-11-03,49,224,...,1,4,4,0,0,0,0,0,0,0
1,2,1982,Graduation,Together,18836.48,0,0,2013-10-13,74,422,...,4,11,6,0,0,0,0,0,0,0
2,3,1951,Graduation,Divorced,66591.34,0,1,2013-12-23,74,157,...,4,10,9,0,0,0,0,0,0,1
3,4,1961,Master,Divorced,40566.58,0,1,2013-11-20,68,568,...,0,3,4,0,0,0,0,0,0,0
4,5,1975,Graduation,Married,59391.42,0,1,2013-09-27,37,133,...,3,5,3,0,0,0,0,0,0,0


## Save the Expanded Dataset

This cell writes the synthetic DataFrame to Parquet. Parquet is compact, fast to read, and suitable for the next stage of the pipeline, including BigQuery ingestion.

In [36]:
output_path = write_synthetic_dataset(
    df_synthetic_campaign,
    SYNTHETIC_OUTPUT_PATH,
    file_format='parquet',
)

print(f'Synthetic dataset written to: {output_path}')

Synthetic dataset written to: ../data/synthetic_marketing_campaign.parquet


## Validate the Synthetic Dataset

This cell checks that the generated dataset has the expected row count, the correct columns, valid dates, no excluded fields, no invalid binary values, and no negative values where they should not exist.

In [39]:
validation_checks = validate_synthetic_campaign_data(
    df_synthetic_campaign,
    df_raw_campaign,
    expected_rows=N_SYNTHETIC_ROWS,
)

validation_checks

{'row_count': 1000000,
 'columns_match': True,
 'unexpected_derived_columns': [],
 'excluded_raw_columns_present': [],
 'expected_row_count_match': True,
 'negative_value_columns': [],
 'invalid_binary_columns': [],
 'date_range_valid': True}

## Compare Customer Group Proportions

This cell compares the original and synthetic proportions for each `Marital_Status` and `Education` combination. The goal is to confirm that the expanded dataset keeps the same customer mix as the cleaned data.

In [42]:
group_proportion_comparison = summarize_group_proportions(
    df_treated_data,
    df_synthetic_campaign,
)

group_proportion_comparison.head(10)

,Marital_Status,Education,original_proportion,synthetic_proportion,difference
0,Divorced,2n Cycle,0.009835,0.009835,4.000894e-07
1,Divorced,Basic,0.000447,0.000447,-2.726866e-08
2,Divorced,Graduation,0.053196,0.053196,-2.449709e-07
3,Divorced,Master,0.016540,0.016540,-8.940545e-09
4,Divorced,PhD,0.023245,0.023245,-4.179705e-07
5,Married,2n Cycle,0.036209,0.036209,-2.087617e-07
6,Married,Basic,0.008941,0.008941,4.546267e-07
7,Married,Graduation,0.193563,0.193563,1.926688e-07
8,Married,Master,0.061690,0.061690,2.369245e-07
9,Married,PhD,0.085829,0.085829,-2.355834e-07


## Compare Key Business Metrics

This cell compares selected original and synthetic averages, such as income, recency, spending, purchases, and campaign response. This gives a quick business-level check that the synthetic data remains close to the original customer behavior.

In [45]:
comparison_columns = [
    'Income',
    'Recency',
    'MntWines',
    'MntMeatProducts',
    'NumWebPurchases',
    'NumStorePurchases',
    'Response',
]

pd.DataFrame(
    {
        'original_mean': df_treated_data[comparison_columns].mean(),
        'synthetic_mean': df_synthetic_campaign[comparison_columns].mean(),
    }
).round(2)

,original_mean,synthetic_mean
Income,51966.64,52052.65
Recency,49.11,49.22
MntWines,304.33,334.09
MntMeatProducts,167.16,195.68
NumWebPurchases,4.09,4.11
NumStorePurchases,5.79,5.87
Response,0.15,0.15


## Next Step: BigQuery and Airflow

The expanded dataset is now ready for the production-style pipeline stage. The next step is to load the Parquet file into BigQuery, then use Airflow to create derived fields such as age, total spend, and purchase totals before running large-scale inference.